# S10: economics and operating model

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cybertide-solutions/ai-platform-architect-labs/blob/main/notebooks/09_economics.ipynb)

**Start:** run the first code cell. It automatically downloads the supporting files in a fresh Colab runtime. No ZIP upload, local installation or pasted setup code is required.

70-minute lab. Turn architecture into an owned service. Use explicit workload and pricing assumptions; do not confuse call quotas with a financial budget.

In [ ]:
# Automatic lab setup: run this cell first. No ZIP upload or manual clone is needed.
from pathlib import Path
import os, sys, json, tempfile, subprocess, shutil

def prepare_lab_files(colab_home=Path('/content')):
    required = ('engine/__init__.py', 'engine/knowledge.py', 'data/policies.json', 'data/orders.json')
    def ready(path):
        return all((path / name).is_file() for name in required)
    try:
        import google.colab
        in_colab = True
    except ImportError:
        in_colab = False
    target = colab_home / 'ai-platform-architect-labs'
    candidates = ([target] if in_colab else []) + [Path.cwd(), Path.cwd().parent]
    root = next((path for path in candidates if ready(path)), None)
    if root is None and in_colab:
        if target.exists():
            raise RuntimeError('An incomplete course folder already exists. Save your work, then disconnect and delete this Colab runtime and reopen the notebook.')
        print('Downloading the course code and data from GitHub...')
        colab_home.mkdir(parents=True, exist_ok=True)
        staging = Path(tempfile.mkdtemp(prefix='course-download-', dir=colab_home))
        try:
            result = subprocess.run(
                ['git', 'clone', '--quiet', '--depth', '1', '--branch', 'main',
                 'https://github.com/cybertide-solutions/ai-platform-architect-labs.git', str(staging / 'repo')],
                capture_output=True, text=True, timeout=120)
            if result.returncode != 0 or not ready(staging / 'repo'):
                raise RuntimeError('Course download failed. Check the runtime internet connection and rerun this cell. Your existing work was not changed.')
            (staging / 'repo').rename(target)
            root = target
        except subprocess.TimeoutExpired:
            raise RuntimeError('Course download timed out. Rerun this cell to try again.') from None
        finally:
            shutil.rmtree(staging, ignore_errors=True)
    if root is None:
        raise RuntimeError('Local Jupyter: open this notebook from the extracted course folder. In Colab, use the README Open in Colab button.')
    root = root.resolve()
    os.chdir(root)
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    print('Lab files ready. Continue with the cells below.')
    return root

ROOT = prepare_lab_files()
# End automatic setup
from engine import read_data
from engine.identity import BUYER, FINANCE, BEACON
from engine.knowledge import Index, evidence_errors
from engine.purchasing import Purchases, query_errors
from engine.platform import Platform
from engine.provider import ModelClient
# Deliberately opt in. No credential is required for software rehearsal.
# Colab only: enable after saving these names in Colab Secrets and granting access.
USE_COLAB_SECRETS=False
if USE_COLAB_SECRETS:
    from google.colab import userdata
    for name in ['CHAT_URL','CHAT_MODEL','CHAT_KEY','EMBED_URL','EMBED_MODEL','EMBED_KEY']:
        os.environ[name]=userdata.get(name)
    os.environ['LIVE_MODE']='1'
LIVE=os.environ.get('LIVE_MODE')=='1'
client=ModelClient.environment() if LIVE else None
print('LIVE MODEL' if LIVE else 'SOFTWARE REHEARSAL: no generated AI answers or AI quality score')

**0–25 min | Calculate a transparent usage scenario.** Prices below are hypothetical currency units per million tokens. Replace them with approved provider quotations and measured token usage. Include retries, cache misses, embeddings, observability, storage, support and availability replicas. No current provider pricing is claimed.

In [ ]:
def token_cost(requests,input_tokens,output_tokens,input_price,output_price,cache_hit=0,retry_fraction=0):
    calls=requests*(1-cache_hit)*(1+retry_fraction)
    return calls*(input_tokens*input_price+output_tokens*output_price)/1_000_000
for requests in [10000,50000,200000]:
    for cache in [0,0.4]:
        print({'monthly_requests':requests,'cache_hit':cache,'hypothetical_token_cost':round(token_cost(requests,1800,300,1,3,cache,.05),2)})
fixed_monthly=600 # Illustrative infrastructure + operations allocation, not a quote.
print('Illustrative hosted total:',fixed_monthly+token_cost(50000,1800,300,1,3,.4,.05))

**25–45 min | Compare serving choices.** Create a table with hosted API, managed private endpoint and self-hosted open model. Include idle capacity, reserved headroom, GPU/runtime operations, data constraints, exit cost and support. A weights-memory estimate alone cannot establish self-hosting viability. State the benchmark and commercial evidence still needed.

**45–60 min | Onboard a third application.** Marketing proposes a visual catalogue assistant. Decide which existing services can be reused, what a multimodal model changes, and who approves new data/model access. Define a service catalogue entry and ownership across platform team, application team, security, data owner and finance.

**60–70 min | Deliver:** cost sensitivity sheet; showback fields; RACI; support/onboarding workflow; 30/60/90-day adoption plan; one explicit decision to defer a capability. Assignment 2 packages this into an individual production-readiness memo.